In [1]:
import os
import cv2
import random
import shutil
import pathlib
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [2]:
CLASS_NAMES = {
    0: "Caries",
    6: "Missing_Teeth",
    7: "Periapical_Lesion",
    11:"Impacted_Tooth",
    13:"Bone_Loss"
}

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [4]:
YOLO_ROOT = pathlib.Path(r"D:\FYP\dental-vision\ML\data\dentaldataset01\YOLO")

ALL_CROPS = pathlib.Path(r"D:\FYP\dental-vision\ML\data\all_crops")

OUTPUT = pathlib.Path(r"D:\FYP\dental-vision\ML\data\processed")

In [5]:
if ALL_CROPS.exists():
    shutil.rmtree(ALL_CROPS)

ALL_CROPS.mkdir(parents=True)

for disease in CLASS_NAMES.values():
    (ALL_CROPS / disease).mkdir(parents=True, exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [6]:
def crop_dataset(split):

    image_dir = YOLO_ROOT / split / "images"
    label_dir = YOLO_ROOT / split / "labels"

    total = 0

    for label_file in tqdm(sorted(label_dir.glob("*.txt"))):

        image_path = None

        for ext in [".jpg",".jpeg",".png",".JPG",".PNG",".JPEG"]:
            candidate = image_dir / (label_file.stem + ext)
            if candidate.exists():
                image_path = candidate
                break

        if image_path is None:
            continue

        image = cv2.imread(str(image_path))

        if image is None:
            continue

        h,w,_ = image.shape

        with open(label_file) as f:

            index = 0

            for line in f:

                values = line.strip().split()

                if len(values)==0:
                    continue

                cls = int(values[0])

                if cls not in CLASS_NAMES:
                    continue

                x,y,bw,bh = map(float, values[1:5])

                xc = x*w
                yc = y*h

                bw *= w
                bh *= h

                pad_x = bw*0.15
                pad_y = bh*0.15

                xmin = int(max(0, xc-bw/2-pad_x))
                xmax = int(min(w, xc+bw/2+pad_x))

                ymin = int(max(0, yc-bh/2-pad_y))
                ymax = int(min(h, yc+bh/2+pad_y))

                crop = image[ymin:ymax, xmin:xmax]

                save_folder = ALL_CROPS / CLASS_NAMES[cls]

                filename = f"{image_path.stem}_{split}_{index}.jpg"

                cv2.imwrite(str(save_folder / filename), crop)

                total += 1
                index += 1

    print(f"{split} : {total} crops")

In [7]:
for split in ["train","valid","test"]:
    crop_dataset(split)

100%|██████████| 9331/9331 [03:29<00:00, 44.58it/s]


train : 34238 crops


100%|██████████| 2871/2871 [01:14<00:00, 38.64it/s]


valid : 10196 crops


100%|██████████| 1730/1730 [00:31<00:00, 55.45it/s]

test : 6194 crops


In [8]:
print(f"{'Disease':25s}Images")

print("-"*40)

for disease in CLASS_NAMES.values():

    total = len(list((ALL_CROPS/disease).glob("*.jpg")))

    print(f"{disease:25s}{total}")

Disease                  Images
----------------------------------------
Caries                   10724
Missing_Teeth            3505
Periapical_Lesion        5291
Impacted_Tooth           27978
Bone_Loss                3130


In [9]:
if OUTPUT.exists():
    shutil.rmtree(OUTPUT)

for split in ["train","valid","test"]:

    for disease in CLASS_NAMES.values():

        (OUTPUT/split/disease).mkdir(parents=True,exist_ok=True)

In [10]:
import shutil
import random
import collections
from sklearn.model_selection import train_test_split

random.seed(42)

OUTPUT = pathlib.Path(r"D:/FYP/dental-vision/ML/data/processed")

# Remove previous processed dataset
if OUTPUT.exists():
    shutil.rmtree(OUTPUT)

OUTPUT.mkdir(parents=True, exist_ok=True)

groups = collections.defaultdict(list)

for class_name in CLASS_NAMES.values():

    crop_dir = ALL_CROPS / class_name

    for crop in crop_dir.glob("*.jpg"):
        
        xray_id = "_".join(crop.stem.split("_")[:-2])

        groups[xray_id].append(crop)

print("Unique X-rays:", len(groups))

Unique X-rays: 13411


In [11]:
multi_crop_groups = {k: v for k, v in groups.items() if len(v) > 1}

print(f"Total groups (unique X-rays): {len(groups)}")
print(f"Groups with more than 1 crop: {len(multi_crop_groups)}")
print(f"Groups with exactly 1 crop: {len(groups) - len(multi_crop_groups)}")

for xray_id, crops in list(multi_crop_groups.items())[:5]:
    print(f"\n{xray_id}  ({len(crops)} crops)")
    for c in crops[:3]:
        print(f"    {c.name}")

if len(multi_crop_groups) == 0:
    print("\n⚠️ WARNING: no multi-crop groups found — grouping is likely still broken. "
          "Do not proceed to the split until this is investigated.")

Total groups (unique X-rays): 13411
Groups with more than 1 crop: 11823
Groups with exactly 1 crop: 1588

000dc27f-NAJIB_MARDANLOO_MASUME_2020-07-12185357_jpg.rf.76cabac5ba504297e98502d2731728d5  (11 crops)
    000dc27f-NAJIB_MARDANLOO_MASUME_2020-07-12185357_jpg.rf.76cabac5ba504297e98502d2731728d5_train_2.jpg
    000dc27f-NAJIB_MARDANLOO_MASUME_2020-07-12185357_jpg.rf.76cabac5ba504297e98502d2731728d5_train_3.jpg
    000dc27f-NAJIB_MARDANLOO_MASUME_2020-07-12185357_jpg.rf.76cabac5ba504297e98502d2731728d5_train_4.jpg

000dc27f-NAJIB_MARDANLOO_MASUME_2020-07-12185357_jpg.rf.8ffbe5dc98f1e8997ea22f5ef5abc337  (8 crops)
    000dc27f-NAJIB_MARDANLOO_MASUME_2020-07-12185357_jpg.rf.8ffbe5dc98f1e8997ea22f5ef5abc337_test_0.jpg
    000dc27f-NAJIB_MARDANLOO_MASUME_2020-07-12185357_jpg.rf.8ffbe5dc98f1e8997ea22f5ef5abc337_test_6.jpg
    000dc27f-NAJIB_MARDANLOO_MASUME_2020-07-12185357_jpg.rf.8ffbe5dc98f1e8997ea22f5ef5abc337_test_5.jpg

000dc27f-NAJIB_MARDANLOO_MASUME_2020-07-12185357_jpg.rf.adab6a52

In [12]:
xray_ids = list(groups.keys())

train_ids, temp_ids = train_test_split(
    xray_ids,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

valid_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

splits = {
    "train": train_ids,
    "valid": valid_ids,
    "test": test_ids
}

for split_name, ids in splits.items():

    for xray_id in ids:

        for crop in groups[xray_id]:

            class_name = crop.parent.name

            destination = OUTPUT / split_name / class_name

            destination.mkdir(parents=True, exist_ok=True)

            shutil.copy2(
                crop,
                destination / crop.name
            )

print("Dataset successfully split by original X-ray.")

Dataset successfully split by original X-ray.


In [13]:
print(f"{'Disease':<22} {'Train':>8} {'Valid':>8} {'Test':>8}")
print("-"*50)

for class_name in CLASS_NAMES.values():

    train_count = len(list((OUTPUT/"train"/class_name).glob("*.jpg")))
    valid_count = len(list((OUTPUT/"valid"/class_name).glob("*.jpg")))
    test_count = len(list((OUTPUT/"test"/class_name).glob("*.jpg")))

    print(f"{class_name:<22} {train_count:>8} {valid_count:>8} {test_count:>8}")

Disease                   Train    Valid     Test
--------------------------------------------------
Caries                     7384     1624     1716
Missing_Teeth              2340      613      552
Periapical_Lesion          3693      806      792
Impacted_Tooth            19775     4087     4116
Bone_Loss                  2182      420      528


In [14]:
for disease in CLASS_NAMES.values():

    train = len(list((OUTPUT/"train"/disease).glob("*.jpg")))
    valid = len(list((OUTPUT/"valid"/disease).glob("*.jpg")))
    test = len(list((OUTPUT/"test"/disease).glob("*.jpg")))

    total = train + valid + test

    print(f"{disease} : {total}")

Caries : 10724
Missing_Teeth : 3505
Periapical_Lesion : 5291
Impacted_Tooth : 27978
Bone_Loss : 3130


In [15]:
def get_xray_ids(split):

    ids = set()

    for cls in CLASS_NAMES.values():

        folder = OUTPUT / split / cls

        if not folder.exists():
            continue

        for img in folder.glob("*.jpg"):
            # Same fix as above: strip trailing "_<split>_<index>", not a nonexistent "_crop_" marker.
            ids.add("_".join(img.stem.split("_")[:-2]))

    return ids

train_ids = get_xray_ids("train")
valid_ids = get_xray_ids("valid")
test_ids = get_xray_ids("test")

print("Train ∩ Valid =", len(train_ids & valid_ids))
print("Train ∩ Test  =", len(train_ids & test_ids))
print("Valid ∩ Test  =", len(valid_ids & test_ids))

Train ∩ Valid = 0
Train ∩ Test  = 0
Valid ∩ Test  = 0


In [16]:
from collections import Counter
import cv2

sizes = Counter()

for disease in CLASS_NAMES.values():
    for img_path in (OUTPUT/"train"/disease).glob("*.jpg"):
        img = cv2.imread(str(img_path))
        if img is not None:
            h, w = img.shape[:2]
            sizes[(w, h)] += 1

print("Unique image sizes:", len(sizes))
print(sizes.most_common(10))

Unique image sizes: 26488
[((482, 504), 8), ((509, 456), 8), ((512, 225), 8), ((443, 491), 8), ((512, 447), 8), ((437, 494), 7), ((504, 489), 7), ((495, 504), 7), ((468, 466), 7), ((494, 496), 7)]
